# Lanatus Systems — Contract Opportunity Mailer

Sends outreach mails **on behalf of the company** offering contract development
teams. For every job row that has a **Contact Email** but no **Contract Mail Sent At**:

1. Researches the target company (DuckDuckGo)
2. Loads the **profile sheet** (`Name / Profile / Experience / URL / Available`) —
   one person can have several profile variants with different experience; Gemini
   picks the **single most suitable variant per person** for this job's role and
   required experience
3. One Gemini call writes the personalized intro + picks the profiles; the rest of
   the mail is a **fixed template** (same structure as our current outreach mail):
   profile table with resume links, team achievements, call to action, signature
4. Sends via Gmail SMTP (plain-text fallback + HTML), stamps the row

## Setup

Reuses `.env` from the other notebooks (`GMAIL_ADDRESS`, `GMAIL_APP_PASSWORD`,
`SHEET_ID`, Gemini keys, `service_account.json`). Add the profile sheet:
```
PROFILE_SHEET_ID=your_profile_sheet_id   # omit if profiles are a tab in the jobs spreadsheet
```
Share the profile sheet with the service-account email too (Viewer is enough).

In [1]:
import os
import sys
from pathlib import Path

# make ../ (updatedlangchain/) importable for the shared `common` package
sys.path.append(str(Path.cwd().parent))

from common.api_key_service import get_google_service

google_service = get_google_service()

SHEET_ID = os.getenv('SHEET_ID')
PROFILE_SHEET_ID = os.getenv('PROFILE_SHEET_ID') or SHEET_ID
PROFILE_WORKSHEET = os.getenv('PROFILE_WORKSHEET')  # None -> first tab
GMAIL_ADDRESS = os.getenv('GMAIL_ADDRESS')
GMAIL_APP_PASSWORD = (os.getenv('GMAIL_APP_PASSWORD') or '').replace(' ', '')

print('Google keys found:', [name for name, _ in google_service.keys])
print('Jobs sheet loaded:', bool(SHEET_ID))
print('Profile sheet:', 'separate' if PROFILE_SHEET_ID != SHEET_ID else 'same spreadsheet')
print('Gmail configured:', bool(GMAIL_ADDRESS and GMAIL_APP_PASSWORD))

Google keys found: ['GOOGLE_API_KEY_1']
Jobs sheet loaded: True
Profile sheet: separate
Gmail configured: True


## Sender identity & team achievements

Edit here when the signature or the achievements list changes.

In [2]:
SENDER = {
    'name': 'Maulik Shah',
    'role': 'Co-Founder',
    'company': 'Lanatus Systems',
    'phone': '+91 9316867779',
    'email': 'maulik@lanatussystems.com',
    'website': 'lanatussystems.com',
    'address': '1107, Shivalik Shilp 2, Opp. ITC Narmada Hotel, Vastrapur, Ahmedabad-380015, Gujarat (India)',
}

TEAM_PITCH = (
    'Our team excels in key tech stacks such as JavaScript (React/Node/Nest/Next), '
    'API Integration, and Cloud (AWS), alongside other robust technologies like '
    'PostgreSQL, MongoDB, and Angular. We are adept at building fast, scalable, '
    'and high-quality engineering solutions.'
)

ACHIEVEMENTS = [
    'Built 10+ web apps using React, Node.js, NestJS, and Next.js, significantly '
    'improving client workflow efficiency by 50%.',
    'Contributed to a SaaS platform supporting over 5,000 active users.',
    'Developed a robust solution for streaming 45+ cameras simultaneously using '
    'S3, EC2, and Node.js.',
    'Achieved a 66% reduction in server costs through database query optimization '
    'and migration to AWS.',
]

# Lanatus brand colors (site primary palette)
ACCENT = '#316bff'        # primary
ACCENT_DARK = '#134cdd'   # primary-400
TINT_1, TINT_2 = '#e9f3ff', '#fbfbfb'  # primary-100 -> near-white gradient

MAPS_URL = (
    'https://www.google.com/maps/place/Shivalik+Shilp+2/@23.0288856,72.5269333,17z/'
    'data=!3m1!4b1!4m6!3m5!1s0x395e84c8d4d4139d:0x177fd3db1f47ada6'
    '!8m2!3d23.0288856!4d72.5295082!16s%2Fg%2F11ckky9nzm?entry=ttu'
)

# real company signature (logo + socials), cleaned of Gmail proxy/markup junk
SIGNATURE_HTML = f"""<table cellpadding="0" cellspacing="0" style="border-collapse:collapse;font-family:Arial,Helvetica,sans-serif;">
<tr>
  <td style="vertical-align:middle;padding:1px;width:73px;text-align:center;">
    <img src="https://d36urhup7zbd7q.cloudfront.net/a/cbcd949b-1754-4fdb-867b-eb870ab0051e.png"
         width="73" height="73" alt="Lanatus Systems"
         style="width:73px;height:73px;vertical-align:middle;border:none;">
  </td>
  <td style="padding:0 0 0 12px;vertical-align:top;">
    <table cellpadding="0" cellspacing="0" style="border-collapse:collapse;">
      <tr><td style="line-height:1.08;padding:0 0 12px;border-bottom:1px solid #212121;">
        <span style="color:#45668e;font-weight:bold;font-size:14px;">{SENDER['name']}</span><br>
        <span style="font-weight:bold;color:#646464;line-height:1.2;font-size:13px;">{SENDER['role']} @ {SENDER['company']}</span>
      </td></tr>
      <tr><td style="padding-top:12px;font-size:11px;color:#212121;line-height:1.2;">
        <a href="tel:+919316867779" style="color:#212121;text-decoration:none;">{SENDER['phone']}</a>
        &nbsp;|&nbsp;
        <a href="https://{SENDER['website']}/" style="color:#212121;">{SENDER['website']}</a>
      </td></tr>
      <tr><td style="padding-top:5px;font-size:11px;line-height:1.2;">
        <a href="mailto:{SENDER['email']}" style="color:#212121;">{SENDER['email']}</a>
      </td></tr>
      <tr><td style="padding-top:5px;font-size:11px;line-height:1.2;">
        <a href="{MAPS_URL}" style="color:#212121;">{SENDER['address']}</a>
      </td></tr>
      <tr><td style="padding-top:12px;">
        <a href="https://m.facebook.com/profile.php?id=100083190858124" style="text-decoration:none;"><img
          src="https://cdn.gifo.wisestamp.com/s/fb/3b5998/48/circle/border.png"
          width="24" height="24" alt="Facebook" style="border:none;"></a>&nbsp;
        <a href="https://www.instagram.com/lanatussystems/" style="text-decoration:none;"><img
          src="https://cdn.gifo.wisestamp.com/s/inst/E4405F/48/circle/border.png"
          width="24" height="24" alt="Instagram" style="border:none;"></a>&nbsp;
        <a href="https://www.linkedin.com/company/lanatus/" style="text-decoration:none;"><img
          src="https://cdn.gifo.wisestamp.com/s/ld/0077b5/48/circle/border.png"
          width="24" height="24" alt="LinkedIn" style="border:none;"></a>
      </td></tr>
    </table>
  </td>
</tr>
</table>"""

## Jobs sheet: rows that still need a contract mail

Separate stamp column (**Contract Mail Sent At**, column L) so this campaign
doesn't clash with the personal application mailer (column K).

In [ ]:
import re
import gspread

COL_JOB_TITLE = 2        # B
COL_COMPANY = 3          # C
COL_LOCATION = 4         # D
COL_APPLY_LINK = 9       # I
COL_CONTACT_EMAIL = 10   # J
COL_CONTRACT_SENT = 12   # L

EMAIL_RE = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')

gc = gspread.service_account(filename='service_account.json')
jobs_ws = gc.open_by_key(SHEET_ID).sheet1

if jobs_ws.cell(1, COL_CONTRACT_SENT).value != 'Contract Mail Sent At':
    jobs_ws.update_cell(1, COL_CONTRACT_SENT, 'Contract Mail Sent At')


def get_pending_jobs() -> list[dict]:
    """Rows with a contact email and no Contract Mail Sent At stamp."""
    pending = []
    for row_idx, row in enumerate(jobs_ws.get_all_values()[1:], start=2):
        row += [''] * (COL_CONTRACT_SENT - len(row))
        emails = EMAIL_RE.findall(row[COL_CONTACT_EMAIL - 1])
        if not emails or row[COL_CONTRACT_SENT - 1].strip():
            continue
        pending.append({
            'row': row_idx,
            'title': row[COL_JOB_TITLE - 1],
            'company': row[COL_COMPANY - 1],
            'location': row[COL_LOCATION - 1],
            'emails': emails,
        })
    return pending


pending = get_pending_jobs()
print(f'{len(pending)} jobs pending contract mail')
for job in pending:
    print(f"  row {job['row']}: {job['title']} @ {job['company']} -> {job['emails']}")

## Profile sheet: available developer profiles

Columns: `Name | Profile | Experience | URL | Available`. One person can appear
several times with different profile variants — selection happens later, per job.

In [ ]:
TRUTHY = {'yes', 'y', 'true', '1', 'available'}


def load_profiles() -> list[dict]:
    """Available profile rows, normalized."""
    book = gc.open_by_key(PROFILE_SHEET_ID)
    ws = book.worksheet(PROFILE_WORKSHEET) if PROFILE_WORKSHEET else book.sheet1
    profiles = []
    for rec in ws.get_all_records():
        rec = {str(k).strip().lower(): v for k, v in rec.items()}
        if str(rec.get('available', '')).strip().lower() not in TRUTHY:
            continue
        exp_match = re.search(r'\d+(?:\.\d+)?', str(rec.get('experience', '')))
        profiles.append({
            'name': str(rec.get('name', '')).strip(),
            'profile': str(rec.get('profile', '')).strip(),
            'experience': float(exp_match.group()) if exp_match else 0,
            'url': str(rec.get('url', '')).strip(),
        })
    return [p for p in profiles if p['name'] and p['url']]


profiles = load_profiles()
print(f'{len(profiles)} available profiles')
for p in profiles:
    print(f"  {p['name']} — {p['profile']} — {p['experience']:g} yrs")

78 available profiles
  Nikhilesh Ramoliya — React,Next — 4 yrs
  Nikhilesh Ramoliya — React,Node,Nest,Postgresql,mongodb,mongo — 4 yrs
  Nikhilesh Ramoliya — React,Node,Supabase — 4 yrs
  Nikhilesh Ramoliya — Node,Nest,Postgresql,mongodb,mongo — 4 yrs
  Nikhilesh Ramoliya — Node,Nest,Postgresql,mongodb,mongo — 5 yrs
  Nikhilesh Ramoliya — React,Next — 5 yrs
  Nikhilesh Ramoliya — React,Node,Supabase — 5 yrs
  Nikhilesh Ramoliya — React,Node,Nest,Postgresql,mongodb,mongo — 5 yrs
  Nikhilesh Ramoliya — React,Node,Supabase — 6 yrs
  Nikhilesh Ramoliya — Node,Nest,Postgresql,mongodb,mongo — 6 yrs
  Nikhilesh Ramoliya — React,Next — 6 yrs
  Nikhilesh Ramoliya — React,Node,Nest,Postgresql,mongodb,mongo — 6 yrs
  Harsh Patel — React,Next — 3 yrs
  Harsh Patel — React,Node,Nest,Node,Nest,Postgresql,mongodb,mongo — 3 yrs
  Harsh Patel — Node,Nest,Postgresql,mongodb,mongo — 3 yrs
  Harsh Patel — Node,Nest,Postgresql,mongodb,mongo — 4 yrs
  Harsh Patel — React,Next — 4 yrs
  Harsh Patel — React,

## Research the target company

In [ ]:
from ddgs import DDGS


def research_company(company: str) -> str:
    """Collect search snippets about the company: what it does, stack, news."""
    queries = [
        f'"{company}" company about',
        f'"{company}" products services technology stack',
    ]
    snippets = []
    with DDGS() as ddgs:
        for query in queries:
            try:
                for hit in ddgs.text(query, max_results=4):
                    snippets.append(f"- {hit['title']}: {hit['body']}")
            except Exception as e:
                print(f'  search failed for {query!r}: {e}')
    seen = set()
    unique = [s for s in snippets if not (s in seen or seen.add(s))]
    return '\n'.join(unique[:10]) or '(no search results found)'

## One Gemini call per job: personalize intro + pick profiles

Returns the two company-specific paragraphs and the indices of the most suitable
profile variants — **at most one variant per person**, matched to the job's role
and seniority. The rest of the mail is template, built in code.

In [ ]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_MODEL = 'gemini-3.1-flash-lite'


class ContractMail(BaseModel):
    """Personalized parts of the contract outreach mail."""
    subject: str = Field(description='Email subject line')
    intro_paragraph: str = Field(description=(
        'First paragraph: who we are + something specific and positive about the '
        'target company from the research'))
    alignment_paragraph: str = Field(description=(
        'Short second paragraph: why our team fits their mission/role'))
    closing_paragraph: str = Field(description=(
        'Paragraph before the call-to-action: how we can support their projects, '
        'referencing the role/company specifics'))
    profile_indices: list[int] = Field(description=(
        '0-based indices of the chosen profile variants, at most one per person'))


def personalize(job: dict, research: str, profiles: list[dict]) -> ContractMail:
    listing = '\n'.join(
        f"{i}. {p['name']} — {p['profile']} — {p['experience']:g} years"
        for i, p in enumerate(profiles)
    )
    prompt = f"""We are {SENDER['company']}, an agency providing skilled development teams for
contract-based collaborations. {SENDER['name']} ({SENDER['role']}) is writing an
outreach mail to the company below, which posted a job — we offer our contract
team instead of a single hire.

TARGET JOB:
- Title: {job['title']}
- Company: {job['company']}
- Location: {job['location'] or 'Remote'}

COMPANY RESEARCH (web snippets — use only facts clearly about this company;
ignore anything that looks like a different company with a similar name):
{research}

OUR TEAM PITCH (context): {TEAM_PITCH}

AVAILABLE PROFILE VARIANTS (one person may appear multiple times with different
experience/role variants):
{listing}

TASKS:
1. subject — professional outreach subject, mentions {SENDER['company']} and the
   kind of team we offer for their role. No clickbait.
2. intro_paragraph — like: 'My name is {SENDER['name']}, and I'm reaching out as
   {SENDER['role']} of {SENDER['company']}. We specialize in providing highly
   skilled development teams for contract-based collaborations, and we've been
   impressed by <specific, REAL facts about the company from the research —
   their products, platform, approach>.' If research is empty/off-topic, praise
   the role's ambitions instead — do NOT invent facts.
3. alignment_paragraph — 1-2 sentences: our capabilities align with their
   mission (reference research if possible).
4. closing_paragraph — 2-3 sentences: how our collective experience can support
   their projects, referencing the role.
5. profile_indices — pick the profiles that best fit this job's tech stack and
   seniority. RULES: at most ONE variant per person (pick the variant whose role
   and experience best match the job); only relevant roles; if the job title
   implies seniority (senior/lead) prefer higher experience variants; 3 to 7
   profiles total.

Tone: professional, confident, concrete. No placeholders, no invented facts."""

    def do_invoke(key: str) -> ContractMail:
        model = ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=key)
        return model.with_structured_output(ContractMail).invoke(prompt)

    result = google_service.call(do_invoke)
    # enforce the one-variant-per-person rule even if the model slips
    chosen, seen_names = [], set()
    for i in result.profile_indices:
        if 0 <= i < len(profiles) and profiles[i]['name'] not in seen_names:
            seen_names.add(profiles[i]['name'])
            chosen.append(i)
    result.profile_indices = chosen
    return result

## Build the mail from the template

Fixed structure in code — only the intro/alignment/closing paragraphs and the
profile table rows change per company. Resume URLs become links in the table.

In [ ]:
from html import escape


def build_html(job: dict, mail: ContractMail, chosen: list[dict]) -> str:
    rows = ''.join(
        f'<tr>'
        f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">{escape(p["name"])}</td>'
        f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">{escape(p["profile"])}</td>'
        f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;text-align:center;">{p["experience"]:g} yrs</td>'
        f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">'
        f'<a href="{escape(p["url"])}" style="color:{ACCENT};">View resume</a></td>'
        f'</tr>'
        for p in chosen
    )
    achievements = ''.join(
        f'<li style="margin-bottom:6px;">{escape(a)}</li>' for a in ACHIEVEMENTS
    )
    return f"""<!DOCTYPE html>
<html><body style="margin:0;padding:24px 0;background-color:{TINT_1};background:linear-gradient(135deg,{TINT_1},{TINT_2});font-family:Arial,Helvetica,sans-serif;">
<table role="presentation" width="100%" cellpadding="0" cellspacing="0"><tr><td align="center">
<table role="presentation" width="720" cellpadding="0" cellspacing="0" style="max-width:720px;background:#ffffff;border-radius:8px;overflow:hidden;">
<tr><td style="background:{ACCENT};padding:18px 28px;">
  <div style="color:#ffffff;font-size:18px;font-weight:bold;">{escape(SENDER['company'])}</div>
  <div style="color:#e9f3ff;font-size:13px;">Contract development teams · {escape(job['title'])}</div>
</td></tr>
<tr><td style="padding:28px;color:#333;font-size:14px;line-height:1.6;">
  <p style="margin-top:0;">Hi {escape(job['company'])} Team,</p>
  <p>{escape(mail.intro_paragraph)}</p>
  <p>{escape(mail.alignment_paragraph)}</p>
  <p style="margin-bottom:6px;"><strong>Developers available for immediate engagement:</strong></p>
  <table role="presentation" width="100%" cellpadding="0" cellspacing="0" style="font-size:13px;border:1px solid #e5e7eb;border-radius:6px;">
    <tr style="background:{TINT_1};">
      <th align="left" style="padding:8px 12px;">Name</th>
      <th align="left" style="padding:8px 12px;">Profile</th>
      <th style="padding:8px 12px;">Experience</th>
      <th align="left" style="padding:8px 12px;">Resume</th>
    </tr>
    {rows}
  </table>
  <p>{escape(TEAM_PITCH)}</p>
  <p style="margin-bottom:6px;"><strong>Notable projects &amp; achievements by our team:</strong></p>
  <ul style="margin-top:0;padding-left:20px;">{achievements}</ul>
  <p>{escape(mail.closing_paragraph)}</p>
  <p>Would you be open to a brief call to discuss how our team can support your
  current and upcoming projects?</p>
  <p style="margin-bottom:16px;">Best regards,</p>
  {SIGNATURE_HTML}
</td></tr>
</table>
</td></tr></table>
</body></html>"""


def build_plain(job: dict, mail: ContractMail, chosen: list[dict]) -> str:
    table = '\n'.join(
        f"- {p['name']} | {p['profile']} | {p['experience']:g} yrs | {p['url']}"
        for p in chosen
    )
    achievements = '\n'.join(f'- {a}' for a in ACHIEVEMENTS)
    return f"""Hi {job['company']} Team,

{mail.intro_paragraph}

{mail.alignment_paragraph}

Developers available for immediate engagement:
{table}

{TEAM_PITCH}

Notable projects & achievements by our team:
{achievements}

{mail.closing_paragraph}

Would you be open to a brief call to discuss how our team can support your
current and upcoming projects?

Best regards,
{SENDER['name']}
{SENDER['role']} @ {SENDER['company']}
{SENDER['phone']} |  {SENDER['website']}

{SENDER['email']}

{SENDER['address']}"""

## Send via Gmail SMTP

In [ ]:
import smtplib
from email.message import EmailMessage


def build_message(to_addrs: list[str], subject: str, plain: str, html: str) -> EmailMessage:
    msg = EmailMessage()
    msg['From'] = f"{SENDER['name']} <{GMAIL_ADDRESS}>"
    msg['To'] = ', '.join(to_addrs)
    msg['Subject'] = subject
    msg.set_content(plain)
    msg.add_alternative(html, subtype='html')
    return msg


def send_mail(msg: EmailMessage) -> None:
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        smtp.send_message(msg)

## Run

**`DRY_RUN = True`** (default) — writes every mail and saves it to
`previews/contract_<company>.html`, sends nothing, stamps nothing. Check the
previews (chosen profiles printed per job), then flip to `False` and re-run.

In [ ]:
from datetime import datetime, timezone

DRY_RUN = False
MAX_MAILS_PER_RUN = 10  # safety cap

PREVIEW_DIR = Path('previews')
PREVIEW_DIR.mkdir(exist_ok=True)

pending = get_pending_jobs()
profiles = load_profiles()
print(f'{len(pending)} pending, {len(profiles)} available profiles, '
      f'sending up to {MAX_MAILS_PER_RUN} (dry run: {DRY_RUN})\n')

sent = 0
for job in pending[:MAX_MAILS_PER_RUN]:
    print(f"=== {job['title']} @ {job['company']} -> {job['emails']} ===")

    research = research_company(job['company'])
    mail = personalize(job, research, profiles)
    chosen = [profiles[i] for i in mail.profile_indices]

    if not chosen:
        print('  no suitable profiles -> skipped\n')
        continue

    print('  Subject:', mail.subject)
    print('  Profiles:', ', '.join(
        f"{p['name']} ({p['profile']}, {p['experience']:g}y)" for p in chosen))

    html = build_html(job, mail, chosen)
    plain = build_plain(job, mail, chosen)

    safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', job['company']).strip('_') or f"row{job['row']}"
    preview = PREVIEW_DIR / f'contract_{safe_name}.html'
    preview.write_text(html, encoding='utf-8')
    print(f'  preview: {preview}')

    if DRY_RUN:
        print()
        continue

    try:
        send_mail(build_message(job['emails'], mail.subject, plain, html))
    except Exception as e:
        print(f'  SEND FAILED: {e}\n')
        continue

    stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')
    jobs_ws.update_cell(job['row'], COL_CONTRACT_SENT, stamp)
    sent += 1
    print(f'  sent + row {job["row"]} stamped\n')

print('Done. Mails sent:', sent)

1 pending, 78 available profiles, sending up to 10 (dry run: False)

=== Fullstack Engineer (MERN) — Contract, Remote @ LaunchDarkly -> ['nikhilesh.r@lanatussystems.com'] ===
  Subject: Extending your MERN stack engineering capacity with Lanatus Systems
  Profiles: Nikhilesh Ramoliya (React,Node,Nest,Postgresql,mongodb,mongo, 6y), Harsh Patel (React,Node,Nest,Node,Nest,Postgresql,mongodb,mongo, 5y), Vipul Lakhara (React,Node,Nest,Postgresql,mongodb,mongo, 6y), Mihir Modi (React,Node,Nest,Postgresql,mongodb,mongo, 6y), Akhil Shah (React,Node,Nest,Postgresql,mongodb,mongo,Angular, 5y), Prakash Dudhat (React,Node,Nest,Postgresql,mongodb,mongo, 9y), Nidhi Patel (React,Node,Nest,Postgresql,mongodb,mongo, 6y)
  preview: previews/contract_LaunchDarkly.html
  sent + row 40 stamped

Done. Mails sent: 1
